# Lorenz-63 Model

The Lorenz-63 model is commonly used to assess various data assimilation methods for nonlinear systems. This model is of interest because of its nonlinear chaotic behavior and low dimensions. The simple three-variable model is composed of a system of nonlinear ordinary differential equations:
\begin{gather*}
	\frac{\partial x}{\partial t} &=& \sigma (y-x) \\
	\frac{\partial y}{\partial t} &=& \rho x-y-xz \\
	\frac{\partial z}{\partial t} &=& xy-\beta z
\end{gather*}
We will use the following common parameter values for the reference solution: $\sigma = 10$, $\rho = 28$, and $\beta = 8/3$. 

<!-- The reference solution will have the initial conditions of $(x_0, y_0, z_0) = (1.508870, -1.531271, 25.46091)$ and a time interval of $t \in [0, 40]$. -->

The observations are generated by adding Gaussian noise with zero mean and a variance of 4.0 to the reference solution and will be measured at regular time intervals $\Delta t = 0.1$.

In [1]:
# LIBRARIES
from main import run_data_assimilation
from models.lorenz63 import Lorenz63
import scipy.io as sio
import numpy as np
from utils.plots import plot_particles, plot_lorenz63, plot_rmses
# import matplotlib.pyplot as plt
from observation_operators import linear_gaussian, LinearGaussian
from filters.kde import KernelDensityEstimation
import ipywidgets # NEEDED TO MAKE interactive plots

data_path = "data/initial/"
T_spinup = 2000
np.random.seed(10) # TODO: remove

# TODO:
# - Save models so we do not have to constantly rerun (Put in an override run ...)
# - Implement cross validation
# - Clean Up ABCs and how they interact with main.py

In [2]:
# Define ensemble sizes to test
# ensemble_sizes = [10, 20, 40, 60, 100, 200, 400, 600]
ensemble_sizes = [10, 20, 40, 60, 100]

RANDOM_SD = 10

# SET UP observation operator
obs_op_params = {
    'mean': 0,
    'std': 2,
    'random_sd': RANDOM_SD
}

observation_op = LinearGaussian(**obs_op_params)

# SET UP filter
sigma_config = {
    "method": "manual",
    "sigma_start": 0.01, 
    "sigma_end": 50, 
    "sigma_obs": 2} # TODO: NOT usually known
filter_params = {
    'sigma_config': sigma_config,
    "sigma_scale": "linear",
    'obs_op': observation_op,
    'random_sd': RANDOM_SD,
    'N_tsteps': 150
    }

filter = KernelDensityEstimation(**filter_params) # Initialize filter

# Dictionaries to store models
models = {}

for ensemble_size in ensemble_sizes:

    # Load data
    data = sio.loadmat(data_path + f"spinup_M{ensemble_size}.mat")['model']
    initial_ensemble = np.array(data['x0'][0][0])

    # SET UP model
    model_params = { # reference dict
        'sigma': 10.0,
        'rho': 28.0,
        'beta': 8.0 / 3.0,
        'for_dt': 0.05, # Process
        'obs_dt': 0.1, 
        'T_steps': 4000, # Number of assimilation steps
        'T_burnin': 2000,
        'ensemble_size': ensemble_size,
        'process_noise': 10e-2, # std PROCESS
        'rand_seed': RANDOM_SD,
        'initial_ensemble': initial_ensemble + np.random.normal(0, np.sqrt(2), initial_ensemble.shape),
        'initial_time': 200, # TODO: Replace with T_spinup 
        'ref_initial_state': np.array(data['xt'][0][0].T)[T_spinup-1], # REF
        'obs_op': observation_op # std # REFERENCE
    }

    model = Lorenz63(**model_params) # Initialize model

    # Run data assimilation
    run_data_assimilation(model, filter) # TODO: If seed is established here?

    # Store model
    models[ensemble_size] = model

Data Assimilation (Ensemble Size=10):   0%|          | 0/4000 [00:00<?, ?step/s]

Data Assimilation (Ensemble Size=20):   0%|          | 0/4000 [00:00<?, ?step/s]

Data Assimilation (Ensemble Size=40):   0%|          | 0/4000 [00:00<?, ?step/s]

Data Assimilation (Ensemble Size=60):   0%|          | 0/4000 [00:00<?, ?step/s]

Data Assimilation (Ensemble Size=100):   0%|          | 0/4000 [00:00<?, ?step/s]

In [4]:
plot_rmses(models, filter)
plot_lorenz63(models, filter)

interactive(children=(Output(),), _dom_classes=('widget-interact',))

interactive(children=(FloatSlider(value=200.0, description='Start Time', max=595.0000000000001, min=200.0, ste…